# 国风炼金卡牌 · 标准批量生成
## 模型：国风4 XL | 负面词：排二次元/3D，不排性别
## 断点续传 · 统一参数 · 数据库驱动

In [ ]:
# @title Cell 1：安装依赖
!pip install diffusers transformers accelerate xformers pillow controlnet-aux huggingface_hub -q
import torch, gc, json, time, hashlib
from pathlib import Path
from PIL import Image, ImageDraw
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from huggingface_hub import hf_hub_download, list_repo_files
from controlnet_aux import CannyDetector
print(f"✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB")

In [ ]:
# @title Cell 2：加载国风4 + ControlNet（首次约8分钟）
# 查文件列表 → 自动下载 → 加载管线
files = list_repo_files("xiaolxl/GuoFeng4_XL")
model_file = [f for f in files if f.endswith(".safetensors")][0]
print(f"📦 模型文件: {model_file}")

model_path = hf_hub_download("xiaolxl/GuoFeng4_XL", model_file, cache_dir="/content/model_cache")
print(f"✅ 已下载: {model_path}")

cn = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_single_file(model_path, controlnet=cn, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to("cuda")
pipe.enable_xformers_memory_efficient_attention()
canny = CannyDetector()
gc.collect(); torch.cuda.empty_cache()
print("✅ 模型就绪")

In [ ]:
# @title Cell 3：加载卡牌数据库 + 提示词模板（标准不可改）
from google.colab import drive
drive.mount('/content/drive')
import json

DRIVE = Path("/content/drive/MyDrive")
CARDS_PATH = DRIVE / "all_cards_export.json"

with open(CARDS_PATH, encoding="utf-8") as f:
    all_cards = json.load(f)
print(f"✅ 卡牌总数: {len(all_cards)}")

# --- 朝代前缀映射 ---
DYN_PREFIX = {"QH":"秦汉","SG":"三国","LJ":"两晋南北朝","ST":"隋唐","SY":"宋元","MQ":"明清"}

def get_dynasty(card):
    cid = card.get("card_id", "")
    pfx = cid.split("-")[0] if "-" in cid else ""
    return DYN_PREFIX.get(pfx, card.get("dynasty", ""))

def get_tags(card):
    tags = card.get("tags", [])
    if isinstance(tags, str):
        try: tags = json.loads(tags)
        except: tags = [t.strip() for t in tags.split(",")]
    return "、".join(tags[:5]) if tags else ""

def get_detail(card):
    for k in ["short_desc", "story", "knowledge_point"]:
        v = card.get(k)
        if v and len(str(v)) > 3:
            return str(v)[:40]
    return ""

# --- 标准提示词模板 ---
TEMPLATES = {
    "person":  "{name}，{dynasty}历史人物，{tags}，中国传统汉服，历史写实风格，国风插画，古代场景背景，{detail}",
    "event":   "{name}，{dynasty}历史事件，{tags}，古代战场，军队将士，历史画风，宏大场景，{detail}",
    "place":   "{name}，{dynasty}历史地标，{tags}，古城建筑，宫殿城墙，无人，国风山水，{detail}",
    "weapon":  "{name}，{dynasty}古代兵器，{tags}，金属冷兵器，深色丝绸背景，静物，无人，{detail}",
    "classic": "{name}，{dynasty}古籍，{tags}，竹简帛书，毛笔，无人，书房静物，{detail}",
    "book":    "{name}，{dynasty}古籍，{tags}，竹简帛书，毛笔，无人，书房静物，{detail}",
    "dynasty": "{name}，{dynasty}王朝象征，{tags}，龙旗玉玺，皇宫，长城，恢宏大气，{detail}",
}

def make_prompt(card):
    t = card.get("type", "person")
    tmpl = TEMPLATES.get(t, TEMPLATES["person"])
    return tmpl.format(
        name=card.get("name", ""),
        dynasty=get_dynasty(card),
        tags=get_tags(card),
        detail=get_detail(card)
    )

# --- 标准负面词（排二次元/3D/西方，不排性别）---
NEG = ("anime, manga, cartoon, chibi, moe, kawaii, "
       "bishounen, bishoujo, big eyes, small nose, pointed chin, "
       "overly feminine, overly masculine, exaggerated muscles, "
       "3d, realistic photo, photograph, photorealistic, "
       "western, european, modern clothing, modern weapon, gun, "
       "nsfw, nude, text, watermark, signature, username, "
       "lowres, worst quality, ugly, deformed, blurry, bad anatomy")

print("✅ 标准就绪")
print(f"   提示词模板: {len(TEMPLATES)} 种类型")
print(f"   负面词长度: {len(NEG)} 字符")
print(f"   示例(第1张): {make_prompt(all_cards[0])[:80]}...")

In [ ]:
# @title Cell 4：单张生成函数
def gen_one(card):
    cid = card["card_id"]
    ctype = card.get("type", "person")
    name = card.get("name", cid)
    
    # 构图参考图（类型特定轮廓）
    ref = Image.new("RGB", (832, 1216), (45, 40, 35))
    draw = ImageDraw.Draw(ref)
    if ctype in ("person", "事件", "event"):
        draw.ellipse((340, 150, 492, 310), outline=(120, 100, 80), width=5)
        draw.rectangle((312, 310, 520, 750), outline=(120, 100, 80), width=5)
    elif ctype in ("place", "地点"):
        draw.rectangle((80, 380, 380, 950), outline=(120, 100, 80), width=5)
        draw.polygon([(60, 380), (230, 160), (400, 380)], outline=(120, 100, 80), width=5)
        draw.rectangle((400, 450, 700, 950), outline=(120, 100, 80), width=5)
    elif ctype in ("weapon", "兵器", "classic", "book", "典籍"):
        draw.line((416, 120, 416, 980), fill=(120, 100, 80), width=6)
        draw.rectangle((340, 900, 492, 970), outline=(120, 100, 80), width=4)
    elif ctype in ("dynasty", "朝代"):
        draw.ellipse((200, 250, 632, 500), outline=(120, 100, 80), width=5)
        draw.rectangle((150, 550, 682, 900), outline=(120, 100, 80), width=5)
    
    edge = canny(ref, low_threshold=50, high_threshold=150)
    prompt = make_prompt(card)
    
    # 固定种子（可复现）
    seed = int(hashlib.md5(cid.encode()).hexdigest()[:8], 16)
    generator = torch.Generator("cuda").manual_seed(seed)
    
    # 统一参数
    result = pipe(
        prompt=prompt, negative_prompt=NEG,
        image=edge, controlnet_conditioning_scale=0.85,
        num_inference_steps=30, guidance_scale=6.5,
        width=832, height=1216, generator=generator
    ).images[0]
    
    return result

print("✅ 生成函数就绪")

In [ ]:
# @title Cell 5：🚀 批量生成（断点续传）
OUT = DRIVE / "card_output_guofeng4"
OUT.mkdir(parents=True, exist_ok=True)

MAX_RETRY = 3
print(f"📋 总数: {len(all_cards)} | 输出: Google Drive/card_output_guofeng4/")
print("=" * 50)

for idx, card in enumerate(all_cards):
    cid = card["card_id"]
    name = card.get("name", cid)
    ctype = card.get("type", "person")
    out_path = OUT / f"{cid}.png"
    
    if out_path.exists():
        continue
    
    print(f"[{idx+1}/{len(all_cards)}] {cid}: {name} ({ctype})")
    
    for attempt in range(1, MAX_RETRY + 1):
        try:
            img = gen_one(card)
            img.save(out_path)
            print(f"  ✅ 已保存")
            gc.collect(); torch.cuda.empty_cache()
            break
        except Exception as e:
            err = str(e)[:60]
            if attempt < MAX_RETRY:
                print(f"  ⚠️ 重试 {attempt}/{MAX_RETRY}: {err}")
                time.sleep(5)
            else:
                print(f"  ❌ 失败: {err}")

done = len(list(OUT.glob("*.png")))
print(f"\n🎉 {done}/{len(all_cards)} 张完成 → Google Drive/card_output_guofeng4/")